In [1]:
from tcgdexsdk import TCGdex, Query
import requests
import asyncio
import pandas as pd
import os


In [2]:
# Init the SDK
tcgdex = TCGdex()

In [3]:
async def fetch_set_prices_full(tcg, set_name):
    set_obj = await tcg.set.get(set_name)
    cards = set_obj.cards

    rows = []

    for c in cards:
        # Fetch full card including pricing
        url = f"https://api.tcgdex.net/v2/en/cards/{c.id}"
        full = requests.get(url).json()

        pricing = full.get("pricing", {})
        cm = pricing.get("cardmarket", {})
        tp = pricing.get("tcgplayer", {})

        # --- Flexible Cardmarket extraction ---
        cm_fields = {}
        for key, value in cm.items():
            # Example: "avg-holo" → "cm_avg_holo"
            safe_key = "cm_" + key.replace("-", "_")
            cm_fields[safe_key] = value

        # --- Flexible TCGPlayer extraction ---
        tp_fields = {
            "tp_updated": tp.get("updated"),
            "tp_unit": tp.get("unit"),
        }

        # Loop through ALL variants dynamically
        for variant_name, variant_data in tp.items():
            if variant_name in ("updated", "unit"):
                continue  # skip metadata fields

            # Example: variant_name = "holofoil"
            prefix = f"tp_{variant_name.replace('-', '_')}"

            for key, value in variant_data.items():
                tp_fields[f"{prefix}_{key}"] = value

        # Build row
        rows.append({
            "id": c.id,
            "name": c.name,
            "localId": c.localId,
            "rarity": full.get("rarity"),
            "supertype": full.get("supertype"),
            **cm_fields,
            **tp_fields,
        })

    return pd.DataFrame(rows)


In [4]:
async def save_daily_snapshot(tcg, set_name, folder="../set_snapshots"):
    # Fetch full pricing dataframe
    df = await fetch_set_prices_full(tcg, set_name)

    # Extract latest Cardmarket update timestamp
    latest_update = df["cm_updated"].dropna().max()
    date_str = pd.to_datetime(latest_update).strftime("%Y-%m-%d")

    # Ensure folder exists
    os.makedirs(folder, exist_ok=True)

    # Build filename
    safe_set_name = set_name.replace(" ", "_").lower()
    filename = f"{safe_set_name}_pricing_{date_str}.csv"
    filepath = os.path.join(folder, filename)

    # Check if file already exists
    if os.path.exists(filepath):
        return f"Skipped saving — file already exists: {filepath}"

    # Save new snapshot
    df.to_csv(filepath, index=False)
    return f"Saved new snapshot: {filepath}"

In [5]:
result = await save_daily_snapshot(tcgdex, "Chaos Rising")
print(result)


Saved new snapshot: ../set_snapshots/chaos_rising_pricing_2026-07-23.csv


In [6]:
result = await save_daily_snapshot(tcgdex, "Pitch Black")
print(result)


Saved new snapshot: ../set_snapshots/pitch_black_pricing_2026-07-23.csv


In [7]:
async def fetch_pokemon_prices_full(tcg, pokemon_name):
    cards = await tcg.card.list(Query().like("name", pokemon_name))

    rows = []

    for c in cards:
        url = f"https://api.tcgdex.net/v2/en/cards/{c.id}"
        full = requests.get(url).json()

        pricing = full.get("pricing", {}) or {}

        # Safe defaults
        cm = pricing.get("cardmarket", {}) or {}
        tp = pricing.get("tcgplayer", {}) or {}

        # --- Flexible Cardmarket extraction ---
        cm_fields = {}
        for key, value in cm.items():
            safe_key = "cm_" + key.replace("-", "_")
            cm_fields[safe_key] = value

        # --- Flexible TCGPlayer extraction ---
        tp_fields = {
            "tp_updated": tp.get("updated"),
            "tp_unit": tp.get("unit"),
        }

        # Loop through ALL variants dynamically
        for variant_name, variant_data in tp.items():
            if variant_name in ("updated", "unit"):
                continue
            if not isinstance(variant_data, dict):
                continue

            prefix = f"tp_{variant_name.replace('-', '_')}"
            for key, value in variant_data.items():
                tp_fields[f"{prefix}_{key}"] = value

        # Build row
        rows.append({
            "id": c.id,
            "name": c.name,
            "localId": c.localId,
            "rarity": full.get("rarity"),
            "supertype": full.get("supertype"),
            "subtypes": full.get("subtypes"),
            "set": full.get("set", {}).get("name"),
            **cm_fields,
            **tp_fields,
        })

    return pd.DataFrame(rows)



In [8]:
async def save_pokemon_snapshot(tcg, pokemon_name, folder="../pokemon_snapshots"):
    df = await fetch_pokemon_prices_full(tcg, pokemon_name)

    # Try Cardmarket timestamps first
    cm_times = df["cm_updated"].dropna()
    tp_times = df["tp_updated"].dropna()

    if len(cm_times) > 0:
        latest_update = cm_times.max()
    elif len(tp_times) > 0:
        latest_update = tp_times.max()
    else:
        latest_update = pd.Timestamp.now().isoformat()

    date_str = pd.to_datetime(latest_update).strftime("%Y-%m-%d")

    os.makedirs(folder, exist_ok=True)

    safe_name = pokemon_name.replace(" ", "_").lower()
    filename = f"{safe_name}_pricing_{date_str}.csv"
    filepath = os.path.join(folder, filename)

    if os.path.exists(filepath):
        return f"Skipped — already exists: {filepath}"

    df.to_csv(filepath, index=False)
    return f"Saved new snapshot: {filepath}"


In [9]:
result = await save_pokemon_snapshot(tcgdex, "Psyduck")
print(result)

Saved new snapshot: ../pokemon_snapshots/psyduck_pricing_2026-07-23.csv


In [10]:
result = await save_pokemon_snapshot(tcgdex, "Pikachu")
print(result)

Saved new snapshot: ../pokemon_snapshots/pikachu_pricing_2026-07-23.csv


In [11]:
result = await save_pokemon_snapshot(tcgdex, "Kadabra")
print(result)

Saved new snapshot: ../pokemon_snapshots/kadabra_pricing_2026-07-23.csv
